# Unidad 2 — Análisis de Componentes Principales (PCA)

> **Inteligencia Computacional · USACH · Prof. Max Chacón**  
> Notebook de ejercicios: cálculo manual + implementación con scikit-learn.

**Temas cubiertos:**
1. PCA desde cero (covarianza → autovalores → scores)
2. PCA con `sklearn` sobre Wisconsin Breast Cancer
3. Scree plot y criterio de Kaiser
4. Biplot (loadings + scores)
5. Reconstrucción y error de compresión

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110
rng = np.random.default_rng(42)

## 1. PCA desde cero — ejemplo 2D (Chacón, Cap. II)

Dataset del capítulo del Prof. Chacón:

In [ ]:
X_raw = np.array([
    [2.5, 2.4], [0.5, 0.7], [2.2, 2.9], [1.9, 2.2], [3.1, 3.0],
    [2.3, 2.7], [2.0, 1.6], [1.0, 1.1], [1.5, 1.6], [1.1, 0.9]
])

# Paso 1: centrar
mu = X_raw.mean(axis=0)
X_c = X_raw - mu
print(f'Medias: {mu}')

# Paso 2: matriz de covarianza
S = np.cov(X_c, rowvar=False)
print(f'\nMatriz de covarianza:\n{np.round(S, 4)}')

# Paso 3: autovalores/autovectores
eigvals, eigvecs = np.linalg.eigh(S)
idx = np.argsort(eigvals)[::-1]          # ordenar descendente
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

print(f'\nAutovalores : {np.round(eigvals, 4)}')
print(f'Autovectores:\n{np.round(eigvecs, 4)}')
var_exp = eigvals / eigvals.sum()
print(f'\nVarianza explicada: PC1={var_exp[0]:.1%}, PC2={var_exp[1]:.1%}')

In [ ]:
# Paso 4: proyección (scores)
Y = X_c @ eigvecs

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(X_raw[:, 0], X_raw[:, 1], s=60, color='steelblue', zorder=3)
for v, c, lbl in zip(eigvecs.T, ['tomato', 'goldenrod'], ['PC1', 'PC2']):
    scale = np.sqrt(eigvals[list(['PC1','PC2']).index(lbl)])
    axes[0].annotate('', xy=mu + scale*v, xytext=mu,
                     arrowprops=dict(arrowstyle='->', color=c, lw=2))
    axes[0].text(*(mu + scale*v + 0.05), lbl, color=c, fontsize=11)
axes[0].set_title('Datos originales + direcciones principales')
axes[0].set_aspect('equal')

axes[1].scatter(Y[:, 0], Y[:, 1], s=60, color='steelblue', zorder=3)
axes[1].axhline(0, color='gray', lw=0.8)
axes[1].axvline(0, color='gray', lw=0.8)
axes[1].set_xlabel(f'PC1 ({var_exp[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({var_exp[1]:.1%})')
axes[1].set_title('Scores (espacio PCA)')
plt.tight_layout()
plt.show()

## 2. PCA con sklearn — Wisconsin Breast Cancer

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_full = PCA().fit(X_scaled)
var_ratio = pca_full.explained_variance_ratio_
var_cum   = np.cumsum(var_ratio)

## 3. Scree plot y criterio de Kaiser

In [ ]:
n_kaiser = (pca_full.explained_variance_ > 1).sum()
n_90     = (var_cum < 0.90).sum() + 1  # componentes para ≥ 90% var

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree plot
axes[0].plot(range(1, len(var_ratio)+1), var_ratio, 'o-', lw=2, color='steelblue')
axes[0].axvline(n_kaiser, color='tomato', ls='--', label=f'Kaiser (n={n_kaiser})')
axes[0].set_xlabel('Componente'); axes[0].set_ylabel('Proporción de varianza')
axes[0].set_title('Scree plot'); axes[0].legend()

# Varianza acumulada
axes[1].plot(range(1, len(var_cum)+1), var_cum, 's-', lw=2, color='goldenrod')
axes[1].axhline(0.90, color='tomato', ls='--', label='90% umbral')
axes[1].axvline(n_90, color='steelblue', ls=':', label=f'{n_90} componentes')
axes[1].set_xlabel('Componente'); axes[1].set_ylabel('Varianza acumulada')
axes[1].set_title('Varianza Explicada Acumulada'); axes[1].legend()

plt.tight_layout(); plt.show()
print(f'Kaiser retiene {n_kaiser} componentes.')
print(f'Para ≥90% varianza se necesitan {n_90} componentes.')

## 4. Biplot (PC1 vs PC2)

In [ ]:
pca2 = PCA(n_components=2).fit(X_scaled)
scores   = pca2.transform(X_scaled)
loadings = pca2.components_.T  # shape (p, 2)

fig, ax = plt.subplots(figsize=(9, 7))

# Scores
colors = ['tomato' if c == 0 else 'steelblue' for c in y]
ax.scatter(scores[:, 0], scores[:, 1], c=colors, s=15, alpha=0.5, zorder=2)

# Loadings escalados
scale = np.sqrt(pca2.explained_variance_) * 3
for i, feat in enumerate(data.feature_names):
    ax.annotate('', xy=(loadings[i, 0]*scale[0], loadings[i, 1]*scale[1]),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='darkgreen', lw=1.2))
    ax.text(loadings[i, 0]*scale[0]*1.05, loadings[i, 1]*scale[1]*1.05,
            feat, fontsize=6.5, color='darkgreen')

handles = [
    mpatches.Patch(color='tomato', label='Maligno'),
    mpatches.Patch(color='steelblue', label='Benigno'),
]
ax.legend(handles=handles)
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})')
ax.set_title('Biplot — Wisconsin Breast Cancer')
plt.tight_layout(); plt.show()

## 5. Reconstrucción y error de compresión

In [ ]:
errors = []
for q in range(1, X_scaled.shape[1] + 1):
    pca_q = PCA(n_components=q).fit(X_scaled)
    X_rec = pca_q.inverse_transform(pca_q.transform(X_scaled))
    mse = np.mean((X_scaled - X_rec) ** 2)
    errors.append(mse)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(errors)+1), errors, 'D-', lw=2, color='purple')
plt.axvline(n_90, color='tomato', ls='--', label=f'{n_90} componentes (90%)')
plt.xlabel('Número de componentes')
plt.ylabel('MSE de reconstrucción')
plt.title('Error de compresión PCA')
plt.legend(); plt.tight_layout(); plt.show()

## Resumen

| Concepto | Resultado (Wisconsin BC) |
|---|---|
| Criterio Kaiser | retiene **5** componentes |
| 90% varianza | alcanzado con **7** componentes |
| PC1+PC2 varianza | ~63% |
| Interpretación PC1 | Dominada por variables de tamaño (radius, perimeter, area) |

> **Ejercicio propuesto**: repetir el análisis **sin** estandarizar y comparar los loadings. ¿Qué variable domina y por qué?